# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. References are made using Croissant `@id`s.

In [ ]:
# List all record sets, fields, and columns by @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets and their Fields:")

for rs in record_sets:
    print(f"- RecordSet: {rs.id}")
    for field in rs.fields:
        print(f"    - Field: {field.id} (column: {field.column.id if field.column else 'N/A'})")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. Reference the record set and field/column `@id`s from the overview above.

In [ ]:
# Choose record sets to extract (using @id)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, select the first record set with actual records
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f'Fields/columns in record set {main_record_set_id}:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record set returned non-empty data.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All fields are referenced by their `@id` names as per schema.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to auto-detect a numeric field by checking each, fallback to user if ambiguous
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    if not numeric_field_id:
        print('No numeric field detected. Please manually inspect and set numeric_field_id.')
    else:
        print(f'Numeric field chosen for analysis: {numeric_field_id}')
        # Use a threshold at the 75th percentile for demonstration
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to auto-detect a categorical field for grouping
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            print(f'Grouping by field: {group_field_id}')
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')
else:
    print('No main record set DataFrame to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field/column references use Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} in {main_record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
In this notebook, we:
- Loaded the dataset and metadata from a Croissant schema URL using `mlcroissant`.
- Explored available record sets, fields, and columns, always referring by their Croissant `@id`s.
- Extracted records for tabular analysis, performed filtering and normalization on a detected numeric field, and grouped by a categorical field.
- Visualized the distribution and group differences in the chosen numeric variable.

This approach can be repeated for other record sets and attributes. For more insights, refer to the dataset's schema or documentation. All code is easily modifiable to reference further fields using their `@id` as needed.